
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lecture: Overview of Hyperparameter Tuning
#### Distributed Model Tuning on Databricks

Why hyperparameter tuning matters in the model-development lifecycle, the core components of a tuning job — the evaluation metric, the hyperparameters and their ranges, and the job configuration — and the main search strategies: grid search, random search, and Bayesian optimization.

<div style="max-width:1180px;margin:20px 0 0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;"><div style="background:#F9F7F4;border:1px solid rgba(27,81,98,0.28);border-left:4px solid #FF5F46;border-radius:10px;padding:18px 22px;"><div style="display:flex;align-items:center;gap:12px;"><svg viewBox="0 0 32 32" width="30" height="30" aria-hidden="true" style="flex:0 0 30px;"><g fill="none" stroke="#FF5F46" stroke-width="1.7"><circle cx="16" cy="16" r="12"/><circle cx="16" cy="16" r="7"/><circle cx="16" cy="16" r="2.4" fill="#FF5F46"/></g></svg><div style="font-size:12px;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;color:#FF5F46;">Learning Objectives</div></div><p style="margin:12px 0 8px;font-size:15px;color:#33414B;">By the end of this lesson, you will be able to:</p><ul style="line-height:1.7;font-size:14.5px;color:#33414B;margin:0;padding-left:22px;"><li style="margin:4px 0;">Explain why hyperparameter tuning matters in the model-development lifecycle.</li><li style="margin:4px 0;">Identify the core components of a tuning job — the evaluation metric, the hyperparameters and their ranges, and the job configuration.</li><li style="margin:4px 0;">Compare the main search strategies: grid search, random search, and Bayesian optimization.</li></ul></div></div>

## Hyperparameter Tuning During Model Development
### Where tuning fits in the workflow

<div style="max-width:1180px;margin:0 auto;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
<p style="font-size:1.02em;line-height:1.65;color:#33414B;margin:0 0 18px;"><b>Purpose:</b> hyperparameter tuning is an <b>inner loop</b> of model development &mdash; it trains many models with different settings, evaluates each, and feeds the best configuration back before the model moves on to evaluation and deployment.</p>
<div style="overflow-x:auto;max-width:1200px;margin:0 auto 4px;">
<svg xmlns="http://www.w3.org/2000/svg" width="100%" viewBox="0 0 1200 260" role="img" style="min-width:900px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <title>Hyperparameter tuning within the model development flow</title>
  <desc>Data preprocessing and feature engineering feed model training and model tuning, which loop back before flowing to model evaluation and model deployment.</desc>
  <defs>
    <marker id="ht-ar" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M1 1 L8 5 L1 9" fill="none" stroke="#1B3139" stroke-width="1.8" stroke-linecap="round"/></marker>
    <marker id="ht-cor" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="8" markerHeight="8" orient="auto-start-reverse"><path d="M1 1 L8 5 L1 9" fill="none" stroke="#FF5F46" stroke-width="2.2" stroke-linecap="round"/></marker>
  </defs>
  <path d="M20 100 h40" stroke="#1B3139" stroke-width="2" fill="none" marker-end="url(#ht-ar)"/>
  <!-- DP/FE -->
  <rect x="66" y="60" width="150" height="90" rx="8" fill="#EEEDE9" stroke="#1B3139"/>
  <text x="141" y="92" text-anchor="middle" font-size="13" font-weight="700" fill="#1B3139">Data</text>
  <text x="141" y="110" text-anchor="middle" font-size="13" font-weight="700" fill="#1B3139">Preprocessing /</text>
  <text x="141" y="128" text-anchor="middle" font-size="13" font-weight="700" fill="#1B3139">Feature Engineering</text>
  <!-- Model Training / Model Tuning stacked -->
  <path d="M216 90 L452 74" stroke="#1B3139" stroke-width="2" fill="none" marker-end="url(#ht-ar)"/>
  <rect x="456" y="52" width="330" height="48" rx="6" fill="#0B2027"/><text x="621" y="83" text-anchor="middle" font-size="20" font-weight="800" fill="#fff">Model Training</text>
  <rect x="456" y="110" width="330" height="48" rx="6" fill="#0B2027"/><text x="621" y="141" text-anchor="middle" font-size="20" font-weight="800" fill="#FF5F46">Model Tuning</text>
  <!-- tuning feedback loop back to DP/FE -->
  <path d="M456 134 C 300 210, 200 200, 141 152" stroke="#FF5F46" stroke-width="2.4" fill="none" marker-end="url(#ht-cor)"/>
  <path d="M786 134 C 880 200, 520 210, 456 138" stroke="#FF5F46" stroke-width="2.4" fill="none"/>
  <!-- Model Evaluation -->
  <path d="M786 76 L968 92" stroke="#1B3139" stroke-width="2" fill="none" marker-end="url(#ht-ar)"/>
  <rect x="972" y="60" width="96" height="90" rx="8" fill="#EEEDE9" stroke="#1B3139"/><text x="1020" y="98" text-anchor="middle" font-size="13" font-weight="700" fill="#1B3139">Model</text><text x="1020" y="116" text-anchor="middle" font-size="13" font-weight="700" fill="#1B3139">Evaluation</text>
  <!-- Model Deployment -->
  <path d="M1068 105 h20" stroke="#1B3139" stroke-width="2" fill="none" marker-end="url(#ht-ar)"/>
  <rect x="1092" y="60" width="96" height="90" rx="8" fill="#EEEDE9" stroke="#1B3139"/><text x="1140" y="98" text-anchor="middle" font-size="13" font-weight="700" fill="#1B3139">Model</text><text x="1140" y="116" text-anchor="middle" font-size="13" font-weight="700" fill="#1B3139">Deployment</text>
</svg>
</div><div style="display:flex;flex-wrap:wrap;gap:16px;margin-top:18px;"><div style="flex:1 1 300px;min-width:280px;box-sizing:border-box;background:#F9F7F4;border:1px solid rgba(27,81,98,0.28);border-radius:10px;padding:16px 18px 18px;"><div style="display:flex;align-items:flex-start;gap:12px;"><svg viewBox="0 0 32 32" width="34" height="34" aria-hidden="true" style="flex:0 0 34px;display:block;"><g fill="none" stroke="#FF5F46" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round"><rect x="4" y="8" width="14" height="10" rx="1.5"/><rect x="8" y="13" width="14" height="10" rx="1.5"/><rect x="12" y="18" width="14" height="10" rx="1.5"/></g></svg><div style="min-width:0;"><div style="font-size:11px;font-weight:700;letter-spacing:0.7px;text-transform:uppercase;color:#FF5F46;line-height:1.3;">What it does</div><div style="font-size:16px;font-weight:700;color:#0b2026;line-height:1.3;margin-top:3px;">Trains many models</div></div></div><p style="margin:12px 0 0;font-size:14.5px;line-height:1.6;color:#33414B;">Tuning fits a model repeatedly, once per hyperparameter combination, to find the best-performing settings.</p></div><div style="flex:1 1 300px;min-width:280px;box-sizing:border-box;background:#F9F7F4;border:1px solid rgba(27,81,98,0.28);border-radius:10px;padding:16px 18px 18px;"><div style="display:flex;align-items:flex-start;gap:12px;"><svg viewBox="0 0 32 32" width="34" height="34" aria-hidden="true" style="flex:0 0 34px;display:block;"><g fill="none" stroke="#FF5F46" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round"><circle cx="16" cy="17" r="11"/><path d="M16 11 v6 l4 3 M11 4 h10"/></g></svg><div style="min-width:0;"><div style="font-size:11px;font-weight:700;letter-spacing:0.7px;text-transform:uppercase;color:#FF5F46;line-height:1.3;">Cost</div><div style="font-size:16px;font-weight:700;color:#0b2026;line-height:1.3;margin-top:3px;">Time-consuming</div></div></div><p style="margin:12px 0 0;font-size:14.5px;line-height:1.6;color:#33414B;">Doing this by hand is slow and error-prone &mdash; it is far better automated by a search strategy.</p></div><div style="flex:1 1 300px;min-width:280px;box-sizing:border-box;background:#F9F7F4;border:1px solid rgba(27,81,98,0.28);border-radius:10px;padding:16px 18px 18px;"><div style="display:flex;align-items:flex-start;gap:12px;"><svg viewBox="0 0 32 32" width="34" height="34" aria-hidden="true" style="flex:0 0 34px;display:block;"><g fill="none" stroke="#FF5F46" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round"><path d="M6 26 V6 M6 26 H28"/><path d="M6 20 L13 14 L19 18 L27 8"/></g><g fill="#FF5F46"><circle cx="13" cy="14" r="1.8"/><circle cx="19" cy="18" r="1.8"/><circle cx="27" cy="8" r="1.8"/></g></svg><div style="min-width:0;"><div style="font-size:11px;font-weight:700;letter-spacing:0.7px;text-transform:uppercase;color:#FF5F46;line-height:1.3;">Trade-off</div><div style="font-size:16px;font-weight:700;color:#0b2026;line-height:1.3;margin-top:3px;">Breadth vs. time</div></div></div><p style="margin:12px 0 0;font-size:14.5px;line-height:1.6;color:#33414B;">The more values you test, the better the result can be &mdash; but the longer the tuning job runs.</p></div></div>
</div>

## Components of Hyperparameter Tuning
### The three things every tuning job needs

<div style="max-width:1180px;margin:0 auto;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
<p style="font-size:1.02em;line-height:1.65;color:#33414B;margin:0 0 18px;"><b>Purpose:</b> every tuning job is defined by three choices &mdash; <b>what</b> you optimize (the metric), <b>which</b> knobs you turn and over what ranges, and <b>how</b> the search runs. Get these right and the search is both effective and efficient.</p>
<div style="display:flex;flex-wrap:wrap;gap:16px;margin-top:0;"><div style="flex:1 1 300px;min-width:280px;box-sizing:border-box;background:#F9F7F4;border:1px solid rgba(27,81,98,0.28);border-radius:10px;padding:16px 18px 18px;"><div style="display:flex;align-items:flex-start;gap:12px;"><svg viewBox="0 0 32 32" width="34" height="34" aria-hidden="true" style="flex:0 0 34px;display:block;"><g fill="none" stroke="#FF5F46" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round"><path d="M4 24 a12 12 0 0 1 24 0"/><path d="M16 24 L23 14"/></g><g fill="#FF5F46"><circle cx="16" cy="24" r="2.2"/></g></svg><div style="min-width:0;"><div style="font-size:11px;font-weight:700;letter-spacing:0.7px;text-transform:uppercase;color:#FF5F46;line-height:1.3;">What to optimize</div><div style="font-size:16px;font-weight:700;color:#0b2026;line-height:1.3;margin-top:3px;">Evaluation Metric</div></div></div><p style="margin:12px 0 0;font-size:14.5px;line-height:1.6;color:#33414B;">The performance measure to improve &mdash; accuracy, F1, precision, recall, RMSE &mdash; chosen to match the ML problem and its success criteria.</p></div><div style="flex:1 1 300px;min-width:280px;box-sizing:border-box;background:#F9F7F4;border:1px solid rgba(27,81,98,0.28);border-radius:10px;padding:16px 18px 18px;"><div style="display:flex;align-items:flex-start;gap:12px;"><svg viewBox="0 0 32 32" width="34" height="34" aria-hidden="true" style="flex:0 0 34px;display:block;"><g fill="none" stroke="#FF5F46" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round"><path d="M6 9 h20 M6 16 h20 M6 23 h20"/></g><g fill="#FF5F46"><circle cx="11" cy="9" r="2.6"/><circle cx="21" cy="16" r="2.6"/><circle cx="14" cy="23" r="2.6"/></g></svg><div style="min-width:0;"><div style="font-size:11px;font-weight:700;letter-spacing:0.7px;text-transform:uppercase;color:#FF5F46;line-height:1.3;">Which knobs</div><div style="font-size:16px;font-weight:700;color:#0b2026;line-height:1.3;margin-top:3px;">Hyperparameters &amp; Ranges</div></div></div><p style="margin:12px 0 0;font-size:14.5px;line-height:1.6;color:#33414B;">The parameters to search (e.g. learning rate, batch size) and their ranges. The algorithm's docs &mdash; and AutoML's best runs &mdash; give good starting points.</p></div><div style="flex:1 1 300px;min-width:280px;box-sizing:border-box;background:#F9F7F4;border:1px solid rgba(27,81,98,0.28);border-radius:10px;padding:16px 18px 18px;"><div style="display:flex;align-items:flex-start;gap:12px;"><svg viewBox="0 0 32 32" width="34" height="34" aria-hidden="true" style="flex:0 0 34px;display:block;"><g fill="none" stroke="#FF5F46" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round"><circle cx="16" cy="16" r="5"/><path d="M16 3 v4 M16 25 v4 M3 16 h4 M25 16 h4 M6.5 6.5 l3 3 M22.5 22.5 l3 3 M25.5 6.5 l-3 3 M9.5 22.5 l-3 3"/></g></svg><div style="min-width:0;"><div style="font-size:11px;font-weight:700;letter-spacing:0.7px;text-transform:uppercase;color:#FF5F46;line-height:1.3;">How to run</div><div style="font-size:16px;font-weight:700;color:#0b2026;line-height:1.3;margin-top:3px;">Job Configuration</div></div></div><p style="margin:12px 0 0;font-size:14.5px;line-height:1.6;color:#33414B;">CPU/GPU use, the search strategy (grid, random, Bayesian), parallelism, stopping criteria and logging. Parallelism increases efficiency.</p></div></div>
</div>

## Optimizing Hyperparameter Values
### Grid, random and Bayesian search

<div style="max-width:1180px;margin:0 auto;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
<p style="font-size:1.02em;line-height:1.65;color:#33414B;margin:0 0 18px;"><b>Purpose:</b> the search strategy decides <b>which</b> hyperparameter combinations to try. They range from exhaustive (grid) to smart and adaptive (Bayesian), trading compute for how quickly they home in on good values.</p>
<div style="overflow-x:auto;max-width:1300px;margin:0 auto 4px;">
<svg xmlns="http://www.w3.org/2000/svg" width="100%" viewBox="0 0 1300 380" role="img" style="min-width:1000px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <title>Hyperparameter search methods</title>
  <desc>Grid search evaluates points on a regular grid; random search samples points at random; Bayesian optimization concentrates sampling near promising regions.</desc>
  <line x1="430" y1="30" x2="430" y2="350" stroke="#E8615A" stroke-width="1" stroke-dasharray="4 4"/>
  <line x1="865" y1="30" x2="865" y2="350" stroke="#E8615A" stroke-width="1" stroke-dasharray="4 4"/>
  <text x="180" y="40" text-anchor="middle" font-size="20" font-weight="800" fill="#1B5162">Grid Search</text><text x="190" y="92" text-anchor="middle" font-size="13" fill="#1c2733">Hyperparameter 1</text><text x="28" y="230" font-size="13" fill="#1c2733" transform="rotate(-90 28 230)" text-anchor="middle">Hyperparameter 2</text><rect x="50" y="100" width="300" height="240" fill="#fff" stroke="#1c2733" stroke-width="1.5"/><circle cx="95" cy="145" r="11" fill="#1c1c1c" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="165" cy="145" r="11" fill="#1c1c1c" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="235" cy="145" r="11" fill="#1c1c1c" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="305" cy="145" r="11" fill="#1c1c1c" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="95" cy="200" r="11" fill="#4d4d4d" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="165" cy="200" r="11" fill="#4d4d4d" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="235" cy="200" r="11" fill="#4d4d4d" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="305" cy="200" r="11" fill="#4d4d4d" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="95" cy="255" r="11" fill="#8a8a8a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="165" cy="255" r="11" fill="#8a8a8a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="235" cy="255" r="11" fill="#8a8a8a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="305" cy="255" r="11" fill="#8a8a8a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="95" cy="310" r="11" fill="#c9d14a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="165" cy="310" r="11" fill="#c9d14a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="235" cy="310" r="11" fill="#e5a02a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="305" cy="310" r="11" fill="#e5462a" stroke="#7a7a7a" stroke-width="0.8"/><text x="615" y="40" text-anchor="middle" font-size="20" font-weight="800" fill="#1B5162">Random Search</text><text x="625" y="92" text-anchor="middle" font-size="13" fill="#1c2733">Hyperparameter 1</text><text x="463" y="230" font-size="13" fill="#1c2733" transform="rotate(-90 463 230)" text-anchor="middle">Hyperparameter 2</text><rect x="485" y="100" width="300" height="240" fill="#fff" stroke="#1c2733" stroke-width="1.5"/><circle cx="545" cy="160" r="11" fill="#1c1c1c" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="595" cy="190" r="11" fill="#333" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="695" cy="170" r="11" fill="#555" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="740" cy="210" r="11" fill="#c7c7c7" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="715" cy="250" r="11" fill="#c7c7c7" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="555" cy="250" r="11" fill="#e5a02a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="635" cy="240" r="11" fill="#888" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="605" cy="300" r="11" fill="#c9d14a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="675" cy="305" r="11" fill="#c7c7c7" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="545" cy="305" r="11" fill="#e5462a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="635" cy="328" r="11" fill="#e5a02a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="730" cy="325" r="11" fill="#333" stroke="#7a7a7a" stroke-width="0.8"/><text x="1050" y="40" text-anchor="middle" font-size="20" font-weight="800" fill="#1B5162">Bayesian Optimization</text><text x="1050" y="64" text-anchor="middle" font-size="12" font-weight="700" fill="#1B3139">The Tree of Parzen Estimators (TPE) is a popular Bayesian method</text><text x="1060" y="92" text-anchor="middle" font-size="13" fill="#1c2733">Hyperparameter 1</text><text x="898" y="230" font-size="13" fill="#1c2733" transform="rotate(-90 898 230)" text-anchor="middle">Hyperparameter 2</text><rect x="920" y="100" width="300" height="240" fill="#fff" stroke="#1c2733" stroke-width="1.5"/><circle cx="980" cy="250" r="11" fill="#1c1c1c" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="1015" cy="220" r="11" fill="#333" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="1040" cy="280" r="11" fill="#888" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="1100" cy="250" r="11" fill="#c9d14a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="1125" cy="235" r="11" fill="#e5a02a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="1120" cy="265" r="11" fill="#e5462a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="1145" cy="250" r="11" fill="#e5a02a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="1155" cy="220" r="11" fill="#c9d14a" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="1175" cy="270" r="11" fill="#888" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="1190" cy="305" r="11" fill="#333" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="1070" cy="310" r="11" fill="#c7c7c7" stroke="#7a7a7a" stroke-width="0.8"/><circle cx="1135" cy="210" r="11" fill="#c7c7c7" stroke="#7a7a7a" stroke-width="0.8"/>
</svg>
</div><div style="display:flex;flex-wrap:wrap;gap:16px;margin-top:18px;"><div style="flex:1 1 300px;min-width:280px;box-sizing:border-box;background:#F9F7F4;border:1px solid rgba(27,81,98,0.28);border-radius:10px;padding:16px 18px 18px;"><div style="display:flex;align-items:flex-start;gap:12px;"><svg viewBox="0 0 32 32" width="34" height="34" aria-hidden="true" style="flex:0 0 34px;display:block;"><g fill="none" stroke="#FF5F46" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round"><rect x="5" y="5" width="22" height="22" rx="1.5"/><path d="M5 12.3 h22 M5 19.6 h22 M12.3 5 v22 M19.6 5 v22"/></g><g fill="#FF5F46"><circle cx="8.6" cy="8.6" r="1.6"/><circle cx="15.9" cy="8.6" r="1.6"/><circle cx="23.2" cy="8.6" r="1.6"/><circle cx="8.6" cy="15.9" r="1.6"/><circle cx="15.9" cy="15.9" r="1.6"/><circle cx="23.2" cy="15.9" r="1.6"/></g></svg><div style="min-width:0;"><div style="font-size:11px;font-weight:700;letter-spacing:0.7px;text-transform:uppercase;color:#FF5F46;line-height:1.3;">Exhaustive</div><div style="font-size:16px;font-weight:700;color:#0b2026;line-height:1.3;margin-top:3px;">Grid Search</div></div></div><p style="margin:12px 0 0;font-size:14.5px;line-height:1.6;color:#33414B;">Evaluates every combination on a regular grid. Simple and thorough, but the cost explodes as parameters and values increase.</p></div><div style="flex:1 1 300px;min-width:280px;box-sizing:border-box;background:#F9F7F4;border:1px solid rgba(27,81,98,0.28);border-radius:10px;padding:16px 18px 18px;"><div style="display:flex;align-items:flex-start;gap:12px;"><svg viewBox="0 0 32 32" width="34" height="34" aria-hidden="true" style="flex:0 0 34px;display:block;"><g fill="none" stroke="#FF5F46" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round"><rect x="5" y="5" width="22" height="22" rx="1.5"/></g><g fill="#FF5F46"><circle cx="10" cy="12" r="1.7"/><circle cx="20" cy="9" r="1.7"/><circle cx="23" cy="18" r="1.7"/><circle cx="14" cy="21" r="1.7"/><circle cx="9" cy="23" r="1.7"/><circle cx="17" cy="15" r="1.7"/></g></svg><div style="min-width:0;"><div style="font-size:11px;font-weight:700;letter-spacing:0.7px;text-transform:uppercase;color:#FF5F46;line-height:1.3;">Sampled</div><div style="font-size:16px;font-weight:700;color:#0b2026;line-height:1.3;margin-top:3px;">Random Search</div></div></div><p style="margin:12px 0 0;font-size:14.5px;line-height:1.6;color:#33414B;">Samples combinations at random. Often finds good values faster than grid search when only a few parameters really matter.</p></div><div style="flex:1 1 300px;min-width:280px;box-sizing:border-box;background:#F9F7F4;border:1px solid rgba(27,81,98,0.28);border-radius:10px;padding:16px 18px 18px;"><div style="display:flex;align-items:flex-start;gap:12px;"><svg viewBox="0 0 32 32" width="34" height="34" aria-hidden="true" style="flex:0 0 34px;display:block;"><g fill="none" stroke="#FF5F46" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round"><path d="M5 26 V5 M5 26 H27"/><path d="M6 24 C 14 22, 16 10, 26 8"/></g><g fill="#FF5F46"><circle cx="20" cy="10" r="1.8"/><circle cx="23" cy="9" r="1.8"/><circle cx="17" cy="13" r="1.8"/></g></svg><div style="min-width:0;"><div style="font-size:11px;font-weight:700;letter-spacing:0.7px;text-transform:uppercase;color:#FF5F46;line-height:1.3;">Adaptive</div><div style="font-size:16px;font-weight:700;color:#0b2026;line-height:1.3;margin-top:3px;">Bayesian Optimization</div></div></div><p style="margin:12px 0 0;font-size:14.5px;line-height:1.6;color:#33414B;">Uses past results to focus sampling on promising regions. The Tree of Parzen Estimators (TPE) is a popular Bayesian method.</p></div></div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>